In [1]:
from torchvision.datasets import FashionMNIST

from dataeval_flow import PipelineConfig, run_tasks
from dataeval_flow.config import (
    DatasetProtocolConfig,
    SourceConfig,
    TaskConfig,
    ViewConfig,
    ViewOperation,
)
from dataeval_flow.config.extractors import BoVWExtractorConfig
from dataeval_flow.workflows.data_cleaning import DataCleaningConfig

# 1. Create the torchvision dataset without transforms. The adapter handles conversion.
tv_dataset = FashionMNIST(root="./data", train=True, download=True)

  0%|          | 0.00/26.4M [00:00<?, ?B/s]

  0%|          | 32.8k/26.4M [00:00<01:53, 233kB/s]

  0%|          | 65.5k/26.4M [00:00<01:54, 231kB/s]

  0%|          | 131k/26.4M [00:00<01:18, 335kB/s] 

  1%|          | 197k/26.4M [00:00<01:08, 384kB/s]

...

 41%|████▏     | 1.84M/4.42M [00:00<00:00, 3.27MB/s]

 83%|████████▎ | 3.67M/4.42M [00:01<00:00, 6.37MB/s]

100%|██████████| 4.42M/4.42M [00:01<00:00, 3.92MB/s]

  0%|          | 0.00/5.15k [00:00<?, ?B/s]

100%|██████████| 5.15k/5.15k [00:00<00:00, 15.8MB/s]

In [2]:
# 2. Build the pipeline configuration with a subset for faster execution.
datasets = [DatasetProtocolConfig(name="fmnist-train", format="torchvision", dataset=tv_dataset)]
# A bare Limit takes the first N samples in storage order.
# Shuffle first so the subset represents the entire dataset.
views = [
    ViewConfig(
        name="sample500",
        operations=[
            ViewOperation(type="Shuffle", params={"seed": 0}),
            ViewOperation(type="Limit", params={"size": 500}),
        ],
    )
]
sources = [SourceConfig(name="fmnist-src", dataset="fmnist-train", view="sample500")]
extractors = [BoVWExtractorConfig(name="bovw", vocab_size=512, batch_size=64)]

workflows = [
    DataCleaningConfig(
        name="adaptive_clean",
        outlier_method="adaptive",
        outlier_threshold=3.5,
        outlier_flags=["dimension", "pixel", "visual"],
    )
]
tasks = [
    TaskConfig(
        name="fmnist-clean",
        workflow="adaptive_clean",
        sources="fmnist-src",
        extractor="bovw",
    )
]

config = PipelineConfig(
    datasets=datasets,
    views=views,
    sources=sources,
    extractors=extractors,
    workflows=workflows,
    tasks=tasks,
)

In [3]:
# 3. Run
results = run_tasks(config)
print(results["fmnist-clean"].report())


  DATA CLEANING COMPLETE. DATASET: 500 ITEMS. MODE: ADVISORY.
  Timestamp:    2026-09-25T21:45:45.666171+00:00
  Duration:     0.99s
  Source:       fmnist-src (fmnist-train[sample500])
  Model:        bovw (bovw)
------------------------------------------------------------------------------------------

  SUMMARY
  -------
  Image Outliers ................................................. 6 images (1.2%)  [..]
  Classwise Outliers ............ worst: T-shirt/top (3.2%), 1/5 classes over 3.0%  [..]
  Duplicates ....................................... 0 exact (0.0%), 9 near (1.8%)  [..]
  Label Distribution ...................... 10 classes, 500 items, imbalance 1.6:1  [..]

  Health: All checks passed [ok]

  IMAGE OUTLIERS                                                           6 images (1.2%)
  6 images (1.2%) flagged as outliers.

  Metric      Count
  ----------  -----
  kurtosis        5
  sharpness       3
  brightness      1

  (Some images trigger multiple metrics.)

  perce

In [4]:
DatasetProtocolConfig(
    name="fmnist-train",
    format="torchvision",
    dataset=tv_dataset,
    version="2",  # bump this when the underlying data changes
)

DatasetProtocolConfig(name='fmnist-train', format='torchvision', dataset=Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train, version='2')